In [1]:
import matplotlib.pyplot as plt
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mansibmursalin/ninapro-db1-full-dataset/Ninapro_DB1.csv


## Data processing

In [2]:
nina=pd.read_csv("/kaggle/input/datasets/mansibmursalin/ninapro-db1-full-dataset/Ninapro_DB1.csv")

In [3]:
nina=nina.drop(columns=['Unnamed: 0','stimulus','repetition'])
nina=nina.drop(columns=nina.filter(like='glove_').columns)


#Remapping restimulus to reduce number of culumns
exercise_map={1:0,2:12,3:29}

#nina['restimulus']=nina['restimulus']+nina['exercise'].map(exercise_map)
mask = nina['restimulus'] != 0
nina.loc[mask, 'restimulus'] = (
    nina.loc[mask, 'restimulus'] + nina.loc[mask, 'exercise'].map(exercise_map)
)
nina=nina.drop(columns='exercise')

In [4]:
# smoothing function
window_size=100
moving_average=np.ones(window_size)/window_size
emg_list=[f'emg_{i}' for i in range(10)]
nina_smooth=nina.copy()

for emg in emg_list:
    nina_smooth[emg]=nina.groupby(['subject','restimulus','rerepetition'])[emg].transform(lambda s: np.convolve(s.to_numpy(),moving_average,mode='same'))


# Best smoothing: 20 or 30. More precision unoticed and not useful

In [5]:
nina_compact=nina.copy()
for emg_num in range(10):
    nina_compact[f'emg_{emg_num}']=nina.groupby(['restimulus','rerepetition','subject'])[f'emg_{emg_num}'].transform(lambda s: np.abs(s.to_numpy()).mean()).to_numpy()

In [6]:
#Removing time component

nina_compact=nina_compact.drop('subject',axis=1).sort_values(by=['rerepetition','restimulus']).drop_duplicates(ignore_index=True)

In [7]:
# MAV done, code for RMS

nina_compact

,emg_0,emg_1,emg_2,emg_3,emg_4,emg_5,emg_6,emg_7,emg_8,emg_9,restimulus,rerepetition
0,0.011950,0.005405,0.005518,0.003892,0.005246,0.019573,0.128275,0.124613,0.012853,0.056837,0,0
1,0.014039,0.025490,0.052654,0.031704,0.003140,0.022253,0.269393,0.143088,0.043874,0.133832,0,0
2,0.087009,0.044021,0.014886,0.004160,0.002558,0.002637,0.038308,0.167672,0.007044,0.107160,0,0
3,0.027085,0.028301,0.004394,0.002698,0.002758,0.038579,0.215315,0.196146,0.004030,0.264526,0,0
4,0.290782,0.046890,0.013568,0.005690,0.003541,0.008754,0.047616,0.173423,0.006874,0.056050,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14062,0.002840,1.005453,0.502617,0.169051,0.153674,0.216142,1.034716,1.086819,0.198069,0.346926,52,10
14063,0.002900,0.326005,0.410785,0.182217,0.176205,0.459358,0.627362,2.578106,0.760059,0.376827,52,10
14064,0.014367,1.278292,0.343039,0.191200,0.271631,1.514314,3.181870,1.737233,0.349481,0.737372,52,10
14065,0.003099,0.201580,0.054212,0.014575,0.018966,0.083152,0.374191,1.271402,0.097463,0.241921,52,10


## ML TIME!!

### KNN

In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import classification_report

def scale_dataset (dataframe, oversample=False):
  X=dataframe[dataframe.columns[:-1]].values # dataframe.columns to access the array of labels
  y=dataframe[dataframe.columns[-1]].values #.values returns the data without the columns (keeps the dimensions)

  scaler = StandardScaler() # Z*=(X-xbar)/std, ! its not necessarily a normal distribution (doesn't work well with outliers since mean and std are affected by extreme values), Reuse the same characteristics on the test set!!
  X=scaler.fit_transform(X)

  if oversample:
    ros=RandomOverSampler() # imbalanced data leads to less recall from the minority class, since the cost function is calculated using all data.
    X,y=ros.fit_resample(X,y) #!duplicates values, so there is a risk of overfitting

  data=np.hstack((X,np.reshape(y,(-1,1)))) # np.hstack (x,y) sticks 2 arrays horizontally, stack needs same dimensions, (-1 means just calculate the other.)

  return data,X,y

In [9]:
# FOR LATER, TRY TO PREDICT REREPETITION

# Train, test, validation
train_mask=nina_compact['rerepetition'].isin(2*n+1 for n in range(5))
train=nina_compact[train_mask]

test_mask=nina_compact['rerepetition'].isin(2*n for n in range(5))
test=nina_compact[test_mask]

# target and prediction
train, X_train,y_train= scale_dataset(train,oversample=True)
test, X_test,y_test= scale_dataset(test,oversample=False)

# KNN
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train,y_train)

# Predict
y_pred = knn_model.predict(X_test)

# score
classification_report(y_test,y_pred)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

'              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00      27.0\n           1       0.00      0.00      0.00       0.0\n           2       0.00      0.00      0.00    1404.0\n           3       0.00      0.00      0.00       0.0\n           4       0.00      0.00      0.00    1404.0\n           5       0.00      0.00      0.00       0.0\n           6       0.00      0.00      0.00    1404.0\n           7       0.00      0.00      0.00       0.0\n           8       0.00      0.00      0.00    1404.0\n           9       0.00      0.00      0.00       0.0\n\n    accuracy                           0.00    5643.0\n   macro avg       0.00      0.00      0.00    5643.0\nweighted avg       0.00      0.00      0.00    5643.0\n'